# BART-Large Evaluation: Baseline vs. GraphRAG

This notebook assumes the final knowledge graph has already been built and saved as:
- `semantic_graph_merged.graphml`
- `semantic_graph_merged.json`

We compare two conditions on the **same** list of people:
1. **Baseline** – BART-large receives only the answer entity.
2. **GraphRAG** – BART-large receives the answer entity + a retrieved subgraph.

In [1]:
!pip install fastcoref spacy transformers torch networkx tqdm pandas tf-keras accelerate>=0.26.0
import os
import re
import json
import unicodedata
import random
from collections import Counter

import networkx as nx
import torch
from transformers import BartTokenizer, BartForConditionalGeneration

I0000 00:00:1782637209.383466    1414 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# =============================================================================
# Load the merged knowledge graph
# =============================================================================

# Adjust path if your graph is in the parent directory
GRAPH_PATH = "semantic_graph_merged.graphml"

print(os.path.getsize("semantic_graph_merged.graphml") / 1024 / 1024, "MB")

if os.path.exists(GRAPH_PATH):
    G = nx.read_graphml(GRAPH_PATH)
else:
    # Fallback: try current working directory
    G = nx.read_graphml("semantic_graph_merged.graphml")

print(f"Loaded graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

21.91071891784668 MB
Loaded graph: 60679 nodes, 55127 edges


In [3]:
# =============================================================================
# Custom list of people to evaluate
# =============================================================================

PEOPLE_LIST = [
    "Albert Einstein",
    "The Beatles",
    "Leonardo da Vinci",
    "Michael Jordan",
    "Marie Curie",
    "William Shakespeare"
]

print(f"Evaluating on {len(PEOPLE_LIST)} people")

Evaluating on 6 people


In [4]:
# =============================================================================
# Load facebook/bart-large (zero-shot, no fine-tuning)
# =============================================================================

BART_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading facebook/bart-large ...")
bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")
bart_model = BartForConditionalGeneration.from_pretrained("facebook/bart-large")
bart_model.to(BART_DEVICE)
bart_model.eval()
print(f"BART loaded on {BART_DEVICE}")

Loading facebook/bart-large ...
BART loaded on cuda


## 1. Baseline – BART with Answer Only (No Knowledge Graph)

In [5]:
# =============================================================================
# Baseline generation: prompt = answer only
# =============================================================================

baseline_results = []

for person in PEOPLE_LIST:
    prompt =f"""
            Answer entity:
            {person}
            
            Write a Jeopardy-style clue.
            
            Generate only the answer.
            """
    inputs = bart_tokenizer(
        prompt,
        return_tensors="pt",
        max_length=64,
        truncation=True
    ).to(BART_DEVICE)

    with torch.no_grad():
        outputs = bart_model.generate(
            **inputs,
            max_length=64,
            num_beams=4,
            early_stopping=True
        )
    clue = bart_tokenizer.decode(outputs[0], skip_special_tokens=True)

    baseline_results.append({
        "person": person,
        "prompt": prompt,
        "generated_clue": clue
    })
    print(f"{person:20s} -> {clue}")

print("\nBaseline evaluation complete. Results stored in `baseline_results`.")

Albert Einstein      -> JEOPARDY-CLUE    Write a Jeopardy-style clue. Answer:   Albert Einstein ______________________________________________   ______   Answer: Albert Einstein.  ______________________________________________________ ______________________________________________________________ ______________________________ Answer: ______ _____________________________ _______________________________
The Beatles          -> Jeopardy-style clue:    The Beatles ______________________________________________ ______________________________________________________________ _______________________________ _____________________________ __________________________ ____________________________ ___________________________ _________________________ _____________________________________ _______________________________________________________________________ ______________________________________________________________________ ____________________________________________________________________ __

## 2. GraphRAG – BART with Answer + Retrieved Subgraph

In [6]:
# =============================================================================
# Subgraph extraction helpers
# =============================================================================

def normalize_entity(text):
    if text is None:
        return ""
    text = unicodedata.normalize("NFKC", str(text))
    text = text.lower()
    text = re.sub(r"<pad>", "", text)
    text = re.sub(r"</?s>", "", text)
    text = re.sub(r"\(.*?\)", "", text)
    text = re.sub(r'"|\'|`', "", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s\-]", "", text)
    return text.strip()


def find_entity_node(graph, entity_name):
    norm_input = normalize_entity(entity_name)
    candidates = []
    for node in graph.nodes():
        norm_node = normalize_entity(node)
        if norm_node == norm_input:
            return node
        if norm_input in norm_node or norm_node in norm_input:
            candidates.append((node, len(norm_node)))
    if candidates:
        candidates.sort(key=lambda x: x[1])
        return candidates[0][0]
    return None


def top_k_confident_edges(graph, node, k=20):
    if node not in graph:
        return []
    edges = []
    for succ in graph.successors(node):
        data = graph[node][succ]
        conf = float(data.get("confidence", 0.0))
        edges.append((node, succ, data, conf))
    edges.sort(key=lambda x: x[3], reverse=True)
    return edges[:k]


def extract_subgraph(entity_name, graph, max_hops=2, top_k=20):
    target = find_entity_node(graph, entity_name)
    if target is None:
        print(f"[subgraph] Entity '{entity_name}' not found in graph.")
        return nx.DiGraph()

    subgraph = nx.DiGraph()
    visited_nodes = {target}
    queue = [(target, 0)]

    while queue:
        current, hop = queue.pop(0)
        if hop >= max_hops:
            continue
        edges = top_k_confident_edges(graph, current, k=top_k)
        for u, v, data, conf in edges:
            if u not in subgraph:
                subgraph.add_node(u, **dict(graph.nodes[u]))
            if v not in subgraph:
                subgraph.add_node(v, **dict(graph.nodes[v]))
            subgraph.add_edge(u, v, **data)
            if v not in visited_nodes:
                visited_nodes.add(v)
                queue.append((v, hop + 1))

    print(f"[subgraph] '{entity_name}' -> {subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges")
    return subgraph


def linearize_subgraph_triples(subgraph):
    triples = []
    for u, v, d in subgraph.edges(data=True):
        rel = d.get("relation", "related_to")
        triples.append(f"{u} {rel} {v}")
    return " | ".join(triples)

print("Subgraph helpers defined.")

Subgraph helpers defined.


In [7]:
# =============================================================================
# GraphRAG generation: prompt = answer + top-20 confident subgraph triples
# =============================================================================

graphrag_results = []

for person in PEOPLE_LIST:
    sg = extract_subgraph(person, graph=G, max_hops=2, top_k=20)
    context = linearize_subgraph_triples(sg)

    if not context:
        print(f"[warning] No subgraph found for '{person}'; falling back to baseline prompt.")
        prompt = f"""
        You are an expert Jeopardy clue writer.
        
        Correct response:
        {person}
        
        Task:
        Write one Jeopardy-style clue whose correct response is the entity above.
        
        Rules:
        - Do NOT mention or reveal the answer.
        - Use only the supporting knowledge.
        - Make the clue concise, natural, and factually accurate.
        - Return only the clue.
        
        Clue:
        """
    else:
        prompt = f"""
        You are an expert Jeopardy clue writer.
        
        Correct response:
        {person}
        
        Supporting knowledge:
        {context}
        
        Task:
        Write one Jeopardy-style clue whose correct response is the entity above.
        
        Rules:
        - Do NOT mention or reveal the answer.
        - Use only the supporting knowledge.
        - Make the clue concise, natural, and factually accurate.
        - Return only the clue.
        
        Clue:
        """
    inputs = bart_tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(BART_DEVICE)

    with torch.no_grad():
        outputs = bart_model.generate(
            **inputs,
            max_length=64,
            num_beams=4,
            early_stopping=True
        )
    clue = bart_tokenizer.decode(outputs[0], skip_special_tokens=True)

    graphrag_results.append({
        "person": person,
        "prompt": prompt,
        "context": context,
        "generated_clue": clue
    })
    print(f"{person:20s} -> {clue}")

print("\nGraphRAG evaluation complete. Results stored in `graphrag_results`.")

[subgraph] 'Albert Einstein' -> 28 nodes, 28 edges
Albert Einstein      -> ジェクト アンドル    ビットピント   You are an expert Jeopardy clue writer. You have the following knowledge and skills: Knowledge of the following entities: ジェコト インドール   オンド
[subgraph] 'The Beatles' -> 6 nodes, 6 edges
The Beatles          -> SourceFile    не  о  ээ  тэрэйчэлэт хэ оэстреч шэкцэ
[subgraph] 'Leonardo da Vinci' -> 23 nodes, 24 edges
Leonardo da Vinci    -> =-=-=-=-=-=-=-=-    You are an expert Jeopardy clue writer.息   チャン   ◄   ▲  ▼  △   ----------------   Task: ▼ ▲ ▼ Clue: △ ▼ ◄ ▼
[subgraph] 'Michael Jordan' -> 3 nodes, 2 edges
Michael Jordan       -> ジャン   அ�ர்தியாலுமைபோகளடங
[subgraph] 'Marie Curie' -> 2 nodes, 1 edges
Marie Curie          -> ジャン   ಠ�ಡಿಲಾಳ್ವರಕನಸಣಮಟಯಾ  
[subgraph] 'William Shakespeare' -> 6 nodes, 5 edges
William Shakespeare  -> SourceFile    - You are an expert Jeopardy-style clue writer.=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-

## 3. Side-by-Side Comparison

In [8]:
# =============================================================================
# Print baseline vs. GraphRAG side-by-side
# =============================================================================

print("=" * 100)
print(f"{'Person':<20} {'Baseline (No KG)':<40} {'GraphRAG (with KG)':<40}")
print("=" * 100)

for b, g in zip(baseline_results, graphrag_results):
    person = b["person"]
    base_clue = b["generated_clue"]
    graph_clue = g["generated_clue"]
    print(f"{person:<20} {base_clue:<40} {graph_clue:<40}")

print("\nDone. You can now inspect `baseline_results` and `graphrag_results` manually.")

Person               Baseline (No KG)                         GraphRAG (with KG)                      
Albert Einstein      JEOPARDY-CLUE    Write a Jeopardy-style clue. Answer:   Albert Einstein ______________________________________________   ______   Answer: Albert Einstein.  ______________________________________________________ ______________________________________________________________ ______________________________ Answer: ______ _____________________________ _______________________________ ジェクト アンドル    ビットピント   You are an expert Jeopardy clue writer. You have the following knowledge and skills: Knowledge of the following entities: ジェコト インドール   オンド
The Beatles          Jeopardy-style clue:    The Beatles ______________________________________________ ______________________________________________________________ _______________________________ _____________________________ __________________________ ____________________________ ___________________________ ____________________

---
## 4. (Optional) Fine-Tune BART on SearchQA & Re-evaluate

Run the cells below if you want to fine-tune BART on SearchQA and then repeat the comparison above with the fine-tuned weights.

In [12]:
# =============================================================================
# Fine-tune BART-large on SearchQA (train only, no eval)
# =============================================================================
from datasets import load_dataset
from transformers import Trainer, TrainingArguments

OUTPUT_DIR = "./bart_searchqa_finetuned"
MAX_TRAIN_SAMPLES = 80000

print("Loading SearchQA ...")
dataset = load_dataset("search_qa", "train_test_val")
train_data = dataset["train"].shuffle(seed=42).select(range(min(MAX_TRAIN_SAMPLES, len(dataset["train"]))))

val_data = dataset["validation"]
test_data = dataset["test"]

def preprocess(example):
    return {
        "source":str(example["answer"]),
        "target":str(example["question"])
    }

train_data = train_data.map(preprocess)
val_data = train_data.map(preprocess)
test_data = train_data.map(preprocess)


train_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")
train_model = BartForConditionalGeneration.from_pretrained("facebook/bart-large")

MAX_INPUT = 32
MAX_TARGET = 64

def tokenize(example):
    model_inputs = train_tokenizer(
        example["source"], truncation=True, max_length=MAX_INPUT, padding="max_length"
    )
    labels = train_tokenizer(
        text_target=example["target"], truncation=True, max_length=MAX_TARGET, padding="max_length"
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tokenized = train_data.map(tokenize, batched=True, remove_columns=train_data.column_names)
val_tokenized = val_data.map(tokenize, batched=True, remove_columns=train_data.column_names)
test_tokenized = test_data.map(tokenize, batched=True, remove_columns=train_data.column_names)

Loading SearchQA ...


Using the latest cached version of the dataset since search_qa couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'train_test_val' at /home/jovyan/.cache/huggingface/datasets/search_qa/train_test_val/1.0.0/8420445ab45ec7c203b01300c14f1516af403a153c319cb2fe71d3132b6016e0 (last modified on Sat Jun 27 20:13:49 2026).


In [10]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    warmup_steps=500,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    save_steps=1000,
    logging_steps=100,
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(model=train_model, args=training_args, train_dataset=train_tokenized)

print("Starting fine-tuning (this will take ~2 hours on GPU) ...")
trainer.train()

train_model.save_pretrained(OUTPUT_DIR)
train_tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Fine-tuned model saved to {OUTPUT_DIR}")

Starting fine-tuning (this will take ~2 hours on GPU) ...


Step,Training Loss
100,15.658800
200,10.280000
300,7.194500
400,4.808100
500,3.200800
600,1.788200
700,1.326600
800,1.259600
900,1.229400
1000,1.217300


/opt/micromamba/lib/python3.11/site-packages/transformers/modeling_utils.py:4037: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Fine-tuned model saved to ./bart_searchqa_finetuned


In [13]:
# =============================================================================
# Re-run GraphRAG evaluation with the fine-tuned model
# =============================================================================

FINE_TUNED_DIR = "./bart_searchqa_finetuned"

print("Loading fine-tuned BART ...")
ft_tokenizer = BartTokenizer.from_pretrained(FINE_TUNED_DIR)
ft_model = BartForConditionalGeneration.from_pretrained(FINE_TUNED_DIR)
ft_model.to(BART_DEVICE)
ft_model.eval()
print("Fine-tuned BART loaded.")

Loading fine-tuned BART ...
Fine-tuned BART loaded.


In [14]:
finetuned_graphrag_results = []

for person in PEOPLE_LIST:
    sg = extract_subgraph(person, graph=G, max_hops=3, top_k=20)
    context = linearize_subgraph_triples(sg)

    if not context:
        prompt = f"""
        You are an expert Jeopardy clue writer.
        
        Correct response:
        {person}
        
        Task:
        Write one Jeopardy-style clue whose correct response is the entity above.
        
        Rules:
        - Do NOT mention or reveal the answer.
        - Use only the supporting knowledge.
        - Make the clue concise, natural, and factually accurate.
        - Return only the clue.
        
        Clue:
        """
    else:
        prompt = f"""
        You are an expert Jeopardy clue writer.
        
        Correct response:
        {person}
        
        Supporting knowledge:
        {context}
        
        Task:
        Write one Jeopardy-style clue whose correct response is the entity above.
        
        Rules:
        - Do NOT mention or reveal the answer.
        - Use only the supporting knowledge.
        - Make the clue concise, natural, and factually accurate.
        - Return only the clue.
        
        Clue:
        """

    inputs = ft_tokenizer(
        prompt, return_tensors="pt", max_length=512, truncation=True
    ).to(BART_DEVICE)

    with torch.no_grad():
        outputs = ft_model.generate(
            **inputs, max_length=64, num_beams=4, early_stopping=True
        )
    clue = ft_tokenizer.decode(outputs[0], skip_special_tokens=True)

    finetuned_graphrag_results.append({
        "person": person,
        "generated_clue": clue
    })
    print(f"{person:20s} -> {clue}")

print("\nFine-tuned GraphRAG evaluation complete.")

[subgraph] 'Albert Einstein' -> 37 nodes, 40 edges
Albert Einstein      -> It's the answer to the question "What's the name of this Nobel Prize-winning physicist?"
[subgraph] 'The Beatles' -> 11 nodes, 14 edges
The Beatles          -> It's the answer to the question "What's the name of this band?"
[subgraph] 'Leonardo da Vinci' -> 32 nodes, 35 edges
Leonardo da Vinci    -> It's the title of the game seen here
[subgraph] 'Michael Jordan' -> 3 nodes, 2 edges
Michael Jordan       -> It's the first clue in the "Jeopardy!" game
[subgraph] 'Marie Curie' -> 2 nodes, 1 edges
Marie Curie          -> It's the answer to the question "What's the name of the person who invented the atomic bomb?"
[subgraph] 'William Shakespeare' -> 6 nodes, 5 edges
William Shakespeare  -> It's the answer to the question "What's the name of this man who wrote "The Tempest"

Fine-tuned GraphRAG evaluation complete.


In [15]:
references = []
predictions = []

batch = val_data[:10000]

for source, target in zip(batch["source"], batch["target"]):
    
    inputs = ft_tokenizer(
        source,
        return_tensors="pt",
        truncation=True,
        max_length=32
    )
    
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = ft_model.generate(
            **inputs,
            num_beams=4,
            max_new_tokens=50
        )
    
    pred = ft_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    references.append(target)
    predictions.append(pred)

print("Exporting")

with open("references.txt","w") as f:
    for q in references:
        f.write(q.strip()+"\n")

with open("hypotheses.txt","w") as f:
    for q in predictions:
        f.write(q.strip()+"\n")

Exporting


In [16]:
import subprocess
import re

cmd = [
    "python",
    "../Answerability-Metric/answerability_score.py",
    "--data_type", "squad",
    "--ref_file", "references.txt",
    "--hyp_file", "hypotheses.txt",
    "--ner_weight", "0.41",
    "--qt_weight", "0.20",
    "--re_weight", "0.36",
    "--delta", "0.66",
    "--ngram_metric", "Bleu_1"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(result.stdout)

# Extract the official scores

text = result.stderr

qbleu1 = float(
    re.search(
        r"Mean Answerability Score Across Questions:\s*([\d.]+)",
        result.stderr
    ).group(1)
)

bleu1 = float(
    re.search(
        r"N-gram Score:\s*([\d.]+)",
        result.stderr
    ).group(1)
)

print(f"Official Q-BLEU-1: {100*qbleu1:.2f}")
print(f"Official BLEU-1: {100*bleu1:.2f}")


Official Q-BLEU-1: 20.20
Official BLEU-1: 15.30
